# GSEA — Clinical Outcome (5Plus vs 2Minus)

Preranked GSEA on the pydeseq2 results stored in `adata.var` of
`bulk_counts_clinical_outcome.h5ad`, looped over every MSigDB database in
`${ACC_DATA_ROOT}/reference/msigdb_gmt`.

Genes are ranked by the Wald `stat`. **Direction:** positive `stat` / NES = higher in
**Good (5Plus)**; negative = higher in **Poor (2Minus)**.

One CSV per database is written to `outputs/gsea/clinical_outcome/`.

> The `.h5ad` is only ever read (`sc.read_h5ad` + `adata.var.copy()`); nothing here writes it back.

In [ ]:
import os
# Project data root. Override with the ACC_DATA_ROOT environment variable.
ACC_DATA_ROOT = os.environ.get("ACC_DATA_ROOT", os.path.expanduser("~/acc_data"))


In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# necessary packages
import os
import time

import pandas as pd
import numpy as np
import gseapy as gp
import scanpy as sc

In [ ]:
H5AD_PATH = f'{ACC_DATA_ROOT}/bulk_rna_data/h5ad_files/clinical_outcome/bulk_counts_clinical_outcome.h5ad'
GMT_DIR = f'{ACC_DATA_ROOT}/reference/msigdb_gmt'
OUT_DIR = f'{ACC_DATA_ROOT}/bulk_rna_data/outputs/gsea/clinical_outcome'

FDR_CUTOFF = 0.05       # significance cutoff used for the summary table
PERMUTATION_NUM = 1000
MIN_SIZE = 15
SEED = 6

os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
adata = sc.read_h5ad(H5AD_PATH)

In [ ]:
adata

## Build the ranking

Same logic as `gsea.ipynb`, except `adata.var` is copied first so the AnnData object is
never mutated in place.

In [ ]:
df = adata.var.copy()

In [ ]:
df.drop(['gene_name', 'gene_biotype', 'description', 'Symbol'], axis=1, inplace=True)
df

In [ ]:
df.dropna(subset=['stat'], inplace=True)
df.sort_values(['stat'], ascending=False, inplace=True)
df

In [ ]:
ranking = df[['gene_label', 'stat']]
ranking.reset_index(drop=True, inplace=True)

# converting all gene names to strings instead of categories.
ranking['gene_label'] = ranking['gene_label'].astype(str)
ranking.drop_duplicates(['gene_label'], inplace=True)
ranking

## Databases

Output names follow the existing `enriched_*.csv` convention already used in `outputs/gsea/`.

In [ ]:
GMT_MAP = {
    'MSigDB_Hallmark.gmt':          'enriched_hallmark.csv',
    'MSigDB_C5_BP.gmt':             'enriched_BP.csv',
    'MSigDB_C5_CC.gmt':             'enriched_CC.csv',
    'MSigDB_C5_MF.gmt':             'enriched_MF.csv',
    'MSigDB_C2_Reactome.gmt':       'enriched_reactome.csv',
    'MSigDB_C2_KEGG_medicus.gmt':   'enriched_KEGG_medicus.csv',
    'MSigDB_C6_Oncogenic_sigs.gmt': 'enriched_oncogenic_sigs.csv',
}

for gmt_file in GMT_MAP:
    assert os.path.exists(os.path.join(GMT_DIR, gmt_file)), gmt_file
len(GMT_MAP)

In [ ]:
def gmt_sizes(gmt_path):
    """Map term -> number of genes in the raw GMT (gseapy only reports the matched size)."""
    sizes = {}
    with open(gmt_path) as fh:
        for line in fh:
            fields = line.rstrip('\n').split('\t')
            if len(fields) > 2:
                sizes[fields[0]] = len([g for g in fields[2:] if g])
    return sizes


def prerank_to_df(pre_res, sizes):
    """Flatten pre_res.results into a tidy DataFrame.

    Term/fdr/es/nes come first and keep their original names so the existing
    bar_plot_gsea_outputs.ipynb reads these files unchanged.
    """
    out = []
    for term, res in pre_res.results.items():
        # gseapy reports 'tag %' as a "lead/matched" string, e.g. "51/144"
        lead_size, matched_size = (int(x) for x in res['tag %'].split('/'))
        out.append([
            term,
            res['fdr'],
            res['es'],
            res['nes'],
            res['pval'],
            sizes.get(term, np.nan),
            matched_size,
            lead_size / matched_size if matched_size else np.nan,
            res['lead_genes'],
        ])

    return (pd.DataFrame(out, columns=['Term', 'fdr', 'es', 'nes', 'pval',
                                       'geneset_size', 'matched_size', 'tag_pct', 'lead_genes'])
              .sort_values('fdr')
              .reset_index(drop=True))

## Run GSEA over every database

GO:BP (7,583 terms) dominates the runtime.

In [ ]:
results = {}

for gmt_file, csv_name in GMT_MAP.items():
    gmt_path = os.path.join(GMT_DIR, gmt_file)
    t0 = time.time()

    pre_res = gp.prerank(rnk=ranking,
                         gene_sets=gmt_path,
                         seed=SEED,
                         threads=4,
                         permutation_num=PERMUTATION_NUM,
                         min_size=MIN_SIZE)

    out_df = prerank_to_df(pre_res, gmt_sizes(gmt_path))
    out_df.to_csv(os.path.join(OUT_DIR, csv_name))
    results[csv_name] = out_df

    n_sig = (out_df['fdr'] < FDR_CUTOFF).sum()
    print(f'{gmt_file:30s} -> {csv_name:28s} {len(out_df):5d} terms | '
          f'{n_sig:4d} at fdr<{FDR_CUTOFF} | {time.time() - t0:6.1f}s')

## Summary

`sig_up_Good` = significant terms enriched in **Good (5Plus)**;
`sig_dn_Poor` = significant terms enriched in **Poor (2Minus)**.

In [ ]:
summary = []
for csv_name, out_df in results.items():
    sig = out_df[out_df['fdr'] < FDR_CUTOFF]
    summary.append([csv_name,
                    len(out_df),
                    len(sig),
                    int((sig['nes'] > 0).sum()),
                    int((sig['nes'] < 0).sum())])

summary_df = pd.DataFrame(summary, columns=['file', 'terms_tested',
                                            f'sig_fdr<{FDR_CUTOFF}', 'sig_up_Good', 'sig_dn_Poor'])
summary_df

In [ ]:
# top significant terms per database, for a quick look
for csv_name, out_df in results.items():
    sig = out_df[out_df['fdr'] < FDR_CUTOFF]
    print(f'\n=== {csv_name} — {len(sig)} significant terms ===')
    if len(sig):
        print(sig[['Term', 'nes', 'fdr', 'matched_size']].head(10).to_string(index=False))
    else:
        print('  (none)')